# Self-Contained Jupyter Notebook: RAG Pipeline for Knowledge Retrieval Agents

This notebook turns the chapter's **most important implementation snippet** into a runnable, self-sufficient notebook.

## Why this snippet?
From Chapter 6, the most central code example is the **RAG pipeline** because it operationalizes the core idea of a **Knowledge Retrieval agent**:
- ingest documents
- chunk them
- build a searchable index
- retrieve relevant passages
- generate a grounded answer

The original chapter example uses **LangChain + OpenAI embeddings + FAISS**. This notebook keeps the same architectural idea, but makes it **self-sufficient** by using:
- pure Python
- `scikit-learn` TF-IDF vectors
- cosine similarity retrieval
- a lightweight grounded answer composer

That means you can run it without API keys.

## Source chapter
Based on the uploaded chapter, especially the **"Example: Building a RAG pipeline"** section on pages 6–7 and the chunking discussion on page 8.

## Notebook goals

By the end of this notebook you will be able to:

1. Load a small document corpus
2. Split it into overlapping chunks
3. Build a vector-style retrieval index
4. Run semantic-style retrieval over the chunks
5. Produce a grounded answer with explicit source passages
6. Experiment with chunk size, overlap, and top-k retrieval

This mirrors the chapter's Knowledge Retrieval workflow:
**Query Understanding -> Retrieval -> Preprocessing -> Reasoning/Generation -> Provenance**

In [ ]:
# Optional: install dependencies if they are not already available
# !pip install scikit-learn pandas numpy

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Tuple
import re
import textwrap
import math

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1) Build a small local corpus

The chapter's original code loads files from a folder.  
To keep this notebook self-contained, we define a small in-memory corpus.

You can later replace this with your own `.txt`, `.md`, PDF-extracted text, policy files, or research notes.

In [ ]:
corpus = {
    "knowledge_retrieval.txt": '''
A Knowledge Retrieval agent connects an LLM's static training data to live or current information.
Its purpose is to reduce hallucination and overcome knowledge cutoff by grounding answers in verifiable sources.

The retrieval process usually has four stages:
1. Query understanding
2. Retrieval from search APIs, vector stores, or databases
3. Preprocessing, such as chunking, filtering, and embedding
4. Reasoning and generation using only the retrieved evidence

A strong retrieval system should preserve provenance, citations, metadata, and traceability.
''',

    "rag_pipeline.txt": '''
A retrieval-augmented generation pipeline first ingests documents, splits them into chunks,
creates vector representations, and stores them in an index. A user query is transformed into
the same vector space, and the top-k most relevant chunks are retrieved.

Those chunks are inserted into the prompt so that the generation step is grounded in evidence.
This improves reliability, transparency, and verifiability.

Chunk size and chunk overlap are critical. Very small chunks improve precision but may lose context.
Very large chunks preserve context but may dilute the signal and hurt retrieval quality.
Overlap helps prevent boundary loss when key facts span two chunks.
''',

    "operations.txt": '''
Production retrieval agents require careful operations management.
Important concerns include noise reduction, index freshness, latency control, and security.

If retrieval quality is poor, inspect the returned source documents.
Typical failure modes include:
- wrong source retrieved
- relevant chunks that still do not answer the question
- vocabulary mismatch between the user query and corpus

Hybrid retrieval can help when semantic retrieval alone misses keyword-specific terminology.
Metadata filters can reduce irrelevant retrieval results.
'''
}

## 2) Chunking utilities

The chapter emphasizes that **chunking is one of the most consequential configuration choices**.
Below we implement overlapping chunking so you can see its effect directly.

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    source: str
    text: str
    start_char: int
    end_char: int

def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def chunk_text(text: str, chunk_size: int = 300, chunk_overlap: int = 60) -> List[Tuple[str, int, int]]:
    text = normalize_whitespace(text)
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end]

        # Try to end on sentence or word boundary for nicer chunks
        if end < n:
            sentence_break = max(chunk.rfind(". "), chunk.rfind("? "), chunk.rfind("! "))
            word_break = chunk.rfind(" ")
            best_break = max(sentence_break, word_break)
            if best_break > chunk_size // 2:
                end = start + best_break + 1
                chunk = text[start:end]

        chunks.append((chunk.strip(), start, end))

        if end == n:
            break

        start = max(0, end - chunk_overlap)

    return chunks

def build_chunks(corpus: Dict[str, str], chunk_size: int = 300, chunk_overlap: int = 60) -> List[Chunk]:
    all_chunks = []
    counter = 1
    for source, text in corpus.items():
        for chunk_text_value, start, end in chunk_text(text, chunk_size=chunk_size, chunk_overlap=chunk_overlap):
            all_chunks.append(
                Chunk(
                    chunk_id=f"C{counter:03d}",
                    source=source,
                    text=chunk_text_value,
                    start_char=start,
                    end_char=end
                )
            )
            counter += 1
    return all_chunks

In [ ]:
chunks = build_chunks(corpus, chunk_size=300, chunk_overlap=60)
print(f"Total chunks created: {len(chunks)}\n")

for c in chunks[:5]:
    print(f"{c.chunk_id} | {c.source} | chars {c.start_char}-{c.end_char}")
    print(textwrap.shorten(c.text, width=120, placeholder=" ..."))
    print("-" * 90)

## 3) Build the retrieval index

The chapter uses embeddings plus a FAISS vector store.  
Here we use **TF-IDF vectors** as a self-contained stand-in for embeddings.

This still demonstrates the essential retrieval pattern:
- convert chunks into vectors
- convert the query into the same vector space
- rank by similarity

In [ ]:
class SimpleVectorIndex:
    def __init__(self, chunks: List[Chunk]):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([c.text for c in chunks])

    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        q_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(q_vec, self.matrix).flatten()
        ranked_idx = np.argsort(sims)[::-1][:top_k]

        results = []
        for idx in ranked_idx:
            results.append({
                "score": float(sims[idx]),
                "chunk": self.chunks[idx]
            })
        return results

In [ ]:
index = SimpleVectorIndex(chunks)

## 4) Create a grounded answer function

In a full production system, this step would call an LLM with the retrieved chunks in context.
Since this notebook is self-contained, we use a simple answer composer that:

- retrieves top chunks
- extracts the most relevant sentences
- returns a grounded response
- includes provenance

In [ ]:
def split_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", normalize_whitespace(text))
    return [p.strip() for p in parts if p.strip()]

def sentence_relevance(sentences: List[str], query: str) -> List[Tuple[str, float]]:
    if not sentences:
        return []
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    X = vec.fit_transform(sentences + [query])
    sent_matrix = X[:-1]
    query_vec = X[-1]
    sims = cosine_similarity(query_vec, sent_matrix).flatten()
    ranked = sorted(zip(sentences, sims), key=lambda x: x[1], reverse=True)
    return ranked

def answer_query(query: str, index: SimpleVectorIndex, top_k: int = 3, max_sentences: int = 4) -> Dict:
    retrieved = index.search(query, top_k=top_k)

    all_candidate_sentences = []
    for item in retrieved:
        chunk = item["chunk"]
        for sent, score in sentence_relevance(split_sentences(chunk.text), query):
            all_candidate_sentences.append({
                "sentence": sent,
                "local_score": float(score),
                "source": chunk.source,
                "chunk_id": chunk.chunk_id,
                "chunk_score": item["score"]
            })

    ranked_sentences = sorted(
        all_candidate_sentences,
        key=lambda x: (x["chunk_score"], x["local_score"]),
        reverse=True
    )

    selected = []
    seen = set()
    for item in ranked_sentences:
        key = item["sentence"].lower()
        if key not in seen and len(item["sentence"]) > 40:
            selected.append(item)
            seen.add(key)
        if len(selected) >= max_sentences:
            break

    answer = " ".join(x["sentence"] for x in selected) if selected else "No grounded answer could be formed."
    sources = [
        {
            "chunk_id": r["chunk"].chunk_id,
            "source": r["chunk"].source,
            "score": round(r["score"], 4),
            "text": r["chunk"].text
        }
        for r in retrieved
    ]

    return {
        "query": query,
        "answer": answer,
        "sources": sources
    }

## 5) Run a query

This mirrors the chapter's sample query execution step.

In [ ]:
query = "What are the main limitations of retrieval-augmented generation?"
result = answer_query(query, index, top_k=3, max_sentences=4)

print("QUESTION")
print(result["query"])
print("\nGROUNDED ANSWER")
print(result["answer"])

print("\nSOURCES")
for s in result["sources"]:
    print(f"- {s['chunk_id']} | {s['source']} | score={s['score']}")

## 6) Inspect the retrieved evidence

This is the notebook equivalent of `return_source_documents=True` in the chapter code.

In [ ]:
for s in result["sources"]:
    print("=" * 100)
    print(f"{s['chunk_id']} | {s['source']} | score={s['score']}")
    print(s["text"])
    print()

## 7) Compare chunking settings

The chapter stresses the trade-off between:
- smaller chunks -> better precision, less context
- larger chunks -> richer context, weaker retrieval focus
- overlap -> reduces boundary loss

Let's compare multiple settings.

In [ ]:
def evaluate_chunk_configs(corpus: Dict[str, str], query: str, configs: List[Tuple[int, int]]) -> pd.DataFrame:
    rows = []
    for chunk_size, chunk_overlap in configs:
        chunks_cfg = build_chunks(corpus, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        idx_cfg = SimpleVectorIndex(chunks_cfg)
        hits = idx_cfg.search(query, top_k=3)

        rows.append({
            "chunk_size": chunk_size,
            "chunk_overlap": chunk_overlap,
            "num_chunks": len(chunks_cfg),
            "top_score": round(hits[0]["score"], 4) if hits else None,
            "top_source": hits[0]["chunk"].source if hits else None,
            "top_chunk_id": hits[0]["chunk"].chunk_id if hits else None
        })
    return pd.DataFrame(rows)

configs = [(180, 30), (250, 50), (300, 60), (500, 80)]
comparison_df = evaluate_chunk_configs(
    corpus,
    "How does chunk overlap affect retrieval quality?",
    configs
)
comparison_df

## 8) A reusable pipeline class

This class wraps the full workflow so you can easily reuse it with your own corpus.

In [ ]:
class MiniRAGPipeline:
    def __init__(self, corpus: Dict[str, str], chunk_size: int = 300, chunk_overlap: int = 60):
        self.corpus = corpus
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.chunks = build_chunks(corpus, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.index = SimpleVectorIndex(self.chunks)

    def ask(self, query: str, top_k: int = 3, max_sentences: int = 4) -> Dict:
        return answer_query(query, self.index, top_k=top_k, max_sentences=max_sentences)

pipeline = MiniRAGPipeline(corpus, chunk_size=300, chunk_overlap=60)

In [ ]:
questions = [
    "Why do retrieval agents reduce hallucination?",
    "Why is provenance important in a retrieval system?",
    "What operational issues matter in production?"
]

for q in questions:
    out = pipeline.ask(q, top_k=3, max_sentences=3)
    print("=" * 100)
    print("Q:", q)
    print("A:", out["answer"])
    print("Sources:", [f"{s['chunk_id']}:{s['source']}" for s in out["sources"]])
    print()

## 9) How this notebook maps to the chapter snippet

### Original chapter snippet
The chapter's RAG example uses:
- `DirectoryLoader`
- `RecursiveCharacterTextSplitter`
- `OpenAIEmbeddings`
- `FAISS`
- `RetrievalQA`
- `ChatOpenAI`

### Self-sufficient notebook version
This notebook replaces those with:
- in-memory local corpus
- custom overlapping chunker
- `TfidfVectorizer`
- cosine similarity search
- lightweight grounded answer composer
- explicit source inspection

### Why this is useful
It preserves the **architecture and reasoning** of the original snippet while removing:
- API-key dependency
- hosted-model dependency
- framework overhead for a learning notebook

## 10) Next upgrade paths

If you want to evolve this notebook toward production, replace components incrementally:

1. **Corpus loading**
   - Load `.txt`, `.md`, PDFs, HTML, or database records

2. **Chunking**
   - Move from fixed-size to recursive or semantic chunking

3. **Embeddings**
   - Replace TF-IDF with sentence embeddings

4. **Index**
   - Replace in-memory matrix search with FAISS, Pinecone, Weaviate, or Milvus

5. **Generation**
   - Replace the heuristic answer composer with an LLM prompt that is constrained to retrieved evidence

6. **Provenance**
   - Add document IDs, page numbers, timestamps, and confidence scores

7. **Operations**
   - Add caching, metadata filtering, freshness workflows, and access control

## 11) Takeaway

The most important snippet in this chapter is the **RAG pipeline** because it expresses the central engineering pattern behind Knowledge Retrieval agents.

This notebook makes that pattern concrete and runnable:
- chunk
- index
- retrieve
- ground
- inspect sources

That is the foundation for more advanced systems later in the chapter, including Document Intelligence and Scientific Research agents.